In [2]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss

from sklearn.datasets import load_breast_cancer

In [3]:
data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target
df.to_csv("breast_cancer.csv", index=False)

df1 = pd.read_csv("breast_cancer.csv")
df1.head(20)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.26540,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.18600,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.24300,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.25750,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.16250,0.2364,0.07678,0
5,12.45,15.70,82.57,477.1,0.12780,0.17000,0.15780,0.08089,0.2087,0.07613,...,23.75,103.40,741.6,0.1791,0.5249,0.5355,0.17410,0.3985,0.12440,0
6,18.25,19.98,119.60,1040.0,0.09463,0.10900,0.11270,0.07400,0.1794,0.05742,...,27.66,153.20,1606.0,0.1442,0.2576,0.3784,0.19320,0.3063,0.08368,0
7,13.71,20.83,90.20,577.9,0.11890,0.16450,0.09366,0.05985,0.2196,0.07451,...,28.14,110.60,897.0,0.1654,0.3682,0.2678,0.15560,0.3196,0.11510,0
8,13.00,21.82,87.50,519.8,0.12730,0.19320,0.18590,0.09353,0.2350,0.07389,...,30.73,106.20,739.3,0.1703,0.5401,0.5390,0.20600,0.4378,0.10720,0
9,12.46,24.04,83.97,475.9,0.11860,0.23960,0.22730,0.08543,0.2030,0.08243,...,40.68,97.65,711.4,0.1853,1.0580,1.1050,0.22100,0.4366,0.20750,0


In [5]:
X = df1.drop("target", axis = 1)
y = df1["target"]
X.shape

(569, 30)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=36, stratify=y)

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=36)

In [10]:
l = np.logspace(-4, 4, 9)
l

array([1.e-04, 1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02, 1.e+03,
       1.e+04])

In [11]:
results = []

In [41]:
for lam in l:
    accuracy_list = []
    for fold, (train_index, val_index) in enumerate(cv.split(X_train, y_train)):

        X_train_fold = X_train.iloc[train_index]
        X_val_fold = X_train.iloc[val_index]

        y_train_fold = y_train.iloc[train_index]
        y_val_fold = y_train.iloc[val_index]

        s = StandardScaler()
        scaled_X_train_fold = s.fit_transform(X_train_fold)
        scaled_X_val_fold = s.transform(X_val_fold)

        model = LogisticRegression(penalty="l2", C=1/lam, max_iter = 1000)
        model.fit(scaled_X_train_fold, y_train_fold)
        y_val_pred = model.predict(scaled_X_val_fold)
        y_val_prob = model.predict_proba(scaled_X_val_fold)
        accuracy = accuracy_score(y_val_fold, y_val_pred)
        loss = log_loss(y_val_fold, y_val_prob)
        accuracy_list.append(accuracy)

    print("lambda = ", lam)
    print("CV accuracy", sum(accuracy_list)/5)
    print("_____________")

lambda =  0.0001
CV accuracy 0.9516483516483516
_____________
lambda =  0.001
CV accuracy 0.945054945054945
_____________
lambda =  0.01
CV accuracy 0.953846153846154
_____________
lambda =  0.1
CV accuracy 0.964835164835165
_____________
lambda =  1.0
CV accuracy 0.9736263736263737
_____________
lambda =  10.0
CV accuracy 0.9714285714285713
_____________
lambda =  100.0
CV accuracy 0.9582417582417582
_____________
lambda =  1000.0
CV accuracy 0.8791208791208792
_____________
lambda =  10000.0
CV accuracy 0.6329670329670329
_____________


In [45]:
s1 = StandardScaler()
scaled_X_train = s1.fit_transform(X_train)
scaled_X_test = s1.transform(X_test)

In [46]:
model1 = LogisticRegression(penalty="l2", C=0.1, max_iter=1000)
model1.fit(scaled_X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.1
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [50]:
y_test_pred = model1.predict(scaled_X_test)
accuracy = accuracy_score(y_test, y_test_pred)
accuracy

0.9912280701754386